#### Imports

In [ ]:
# Imports
import os
import sys
import torch
from tqdm import tqdm
import supervision as sv
#from inference import get_model
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor

#### Path Configurations

In [ ]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..','..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
result_directory = os.path.join(project_root, "results")

# Model Paths
checkpoint_number = 2384
base_model_name = "PekingU/rtdetr_v2_r101vd"
model_name = "28-04-2026_21-08_rtdetr_v2_r101vd"
models_directory = os.path.join(project_root, "models", "detection")
full_model_weights_path = os.path.join(models_directory, model_name, f"checkpoint-{checkpoint_number}")

# Test File
file_name = "121364_0"
dataset_name = "roboflow_samples"
data_directory = os.path.join(project_root, "data")
raw_video_data_directory = os.path.join(data_directory, "raw_videos")
full_video_file_path = os.path.join(raw_video_data_directory, dataset_name, f"{file_name}.mp4")

# Results Path
full_result_output_path = os.path.join(result_directory, f"{file_name}_{model_name}_detections.mp4")

#### Image Processor and Model Setup

In [ ]:
# Get mapping from category id to category name
categories = ['ball', 'goalkeeper', 'player', 'referee']
id2label = {index: x for index, x in enumerate(categories, start=0)}
label2id = {v: k for k, v in id2label.items()}

# Image Processor
image_processor = RTDetrImageProcessor.from_pretrained(
    base_model_name,
    do_resize=True,
    size={"width": 1280, "height": 1280},
    use_Fast=True,
)

# Detection Model
detection_model = RTDetrV2ForObjectDetection.from_pretrained(
    full_model_weights_path,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # needed when num_labels differs from the checkpoint  
    local_files_only=True,  
)

Loading weights: 100%|██████████| 1025/1025 [00:00<00:00, 18961.01it/s]


#### Ball, Goalkeeper, Players, Referees Coloring

In [ ]:
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000')
)


#### Video Annotation Loop

In [ ]:
# Video Annotation Setup
video_info = sv.VideoInfo.from_video_path(full_video_file_path)
video_sink = sv.VideoSink(full_result_output_path, video_info=video_info)
frame_generator = sv.get_video_frames_generator(full_video_file_path)

# id2label comes from your model config, e.g. detection_model.config.id2label
id2label = detection_model.config.id2label

# Video Annotation Loop
with video_sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        h, w = frame.shape[:2]  # frame is a numpy array

        inputs = image_processor(images=frame, return_tensors="pt")
        with torch.no_grad():
            outputs = detection_model(**inputs)

        postprocessed_outputs = image_processor.post_process_object_detection(
            outputs,
            target_sizes=[(h, w)],
            threshold=0.1, # Would typically use 0.3, but confience is low with a small training dataset
        )
        image_detections = postprocessed_outputs[0]

        # Build sv.Detections from the HuggingFace output
        boxes = image_detections["boxes"].cpu().numpy()       # (N, 4) xyxy
        scores = image_detections["scores"].cpu().numpy()     # (N,)
        class_ids = image_detections["labels"].cpu().numpy()  # (N,)

        detections = sv.Detections(
            xyxy=boxes,
            confidence=scores,
            class_id=class_ids,
        )

        labels = [
            f"{id2label[class_id]} {confidence:.2f}"
            for class_id, confidence
            in zip(class_ids, scores)
        ]

        annotated_frame = frame.copy()
        annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
        annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
        video_sink.write_frame(annotated_frame)

  0%|          | 0/750 [00:00<?, ?it/s]

  0%|          | 0/750 [00:02<?, ?it/s]

RTDetrV2ObjectDetectionOutput(loss=None, loss_dict=None, logits=tensor([[[-5.0096, -4.5718, -3.3331, -4.8529],
         [-5.0547, -4.6864, -3.7701, -4.9531],
         [-4.8052, -4.4433, -2.9816, -5.0098],
         ...,
         [-5.4878, -5.8143, -5.8629, -5.6912],
         [-5.6014, -5.5507, -5.5614, -5.3990],
         [-5.6494, -5.6129, -5.6043, -5.4930]]]), pred_boxes=tensor([[[0.3599, 0.4478, 0.0168, 0.0442],
         [0.7438, 0.4532, 0.0169, 0.0450],
         [0.7286, 0.5597, 0.0231, 0.0523],
         ...,
         [0.1450, 0.8101, 0.1706, 0.1504],
         [0.0642, 0.3296, 0.1205, 0.2399],
         [0.4315, 0.0546, 0.1299, 0.0863]]]), auxiliary_outputs=None, last_hidden_state=tensor([[[-4.4842e-01,  1.5070e-01, -1.1732e-01,  ...,  1.0994e+00,
          -3.0205e-03,  1.3802e+00],
         [-5.6776e-01,  2.0364e-02, -2.6915e-01,  ...,  1.3907e+00,
           1.4158e-01,  1.3021e+00],
         [-4.6869e-01,  2.6589e-01, -1.9526e-01,  ...,  8.2181e-01,
          -1.3981e-02,  1.2937e